# Topic 1: DataFrame vs RDD

> **Phase 3 → Week 3 → Topic 1**
>
> Prerequisites: Week 2 complete (RDDs — all 5 notebooks)

---

## The GPS vs Paper Map Analogy

Using an **RDD** is like navigating with a paper map:
- You control every single step (turn left, go 2km, turn right)
- Very flexible — you can take any route
- But YOU must figure out the optimal path yourself
- No auto-rerouting if there's traffic

Using a **DataFrame** is like using GPS:
- You just say the destination: `groupBy("city").count()`
- The GPS (Catalyst Optimizer) figures out the optimal route automatically
- It knows about shortcuts (predicate pushdown, column pruning)
- Adapts to conditions (AQE) — faster, less mental overhead

**Same destination. DataFrame gets there smarter.**

---

## What is a DataFrame?

A **DataFrame** is a distributed collection of data organized into **named columns** — like a table in a database or a Pandas DataFrame, but distributed across a cluster.

```
RDD:        [Row1_raw, Row2_raw, Row3_raw, ...]   ← just values, no structure
DataFrame:  ┌────┬──────┬─────────────┬────────┐
            │ id │ name │ dept        │ salary │  ← named, typed columns
            ├────┼──────┼─────────────┼────────┤
            │  1 │Alice │ Engineering │  95000 │
            │  2 │ Bob  │ Engineering │  88000 │
            └────┴──────┴─────────────┴────────┘
```

### DataFrame = RDD + Schema + Catalyst Optimizer

```
RDD            +     Schema         +   Catalyst Optimizer
(distributed       (column names          (query optimization:
 collection)        and types)             predicate pushdown,
                                           column pruning,
                                           join reordering)
     └──────────────────┴──────────────────┘
                         │
                    DataFrame
```

Under the hood, a DataFrame IS an RDD of `Row` objects — but with extra intelligence on top.

---

## Why DataFrames Are Faster Than RDDs

### 1. Catalyst Optimizer — Automatic Query Rewriting

With RDD, YOU write the execution plan. With DataFrame, Catalyst rewrites it for you.

```python
# RDD (you decide the order — wrong order = slow)
rdd.join(big_rdd)          # join millions of rows
   .filter(country=='IN')  # filter AFTER join (too late!)

# DataFrame (Catalyst reorders it automatically)
df.join(big_df, 'id')      # Catalyst moves filter BEFORE join
  .filter(country=='IN')   # predicate pushdown = filter early
```

### 2. Tungsten Execution Engine — Binary Memory Format

- RDD uses standard Python objects → stored as Python objects in JVM heap → heavy GC
- DataFrame uses Tungsten's off-heap binary format → much less GC, CPU cache-friendly
- Tungsten generates optimized JVM bytecode at runtime (whole-stage code generation)

### 3. Column Pruning — Don't Read What You Don't Need

```python
# RDD: read all 50 columns from Parquet, then select 2
rdd.map(lambda row: (row.name, row.salary))

# DataFrame: Catalyst tells Parquet reader to skip the other 48 columns
df.select('name', 'salary')  # only 2 columns read from disk!
```

### 4. Unified Across Languages

RDD performance varies between Python/Java/Scala (Python RDD = serialization overhead).
DataFrame has the SAME performance in all languages because execution happens in JVM regardless.

---

## DataFrame vs RDD — Decision Guide

| Use **DataFrame** when... | Use **RDD** when... |
|--------------------------|--------------------|
| Working with structured/semi-structured data (CSV, JSON, Parquet) | Truly unstructured data (binary files, custom formats) |
| SQL-style operations (select, groupBy, join) | Complex custom transformations not expressible in DataFrame API |
| Performance is critical (Catalyst + Tungsten) | You need fine-grained control over partitioning |
| Interoperability with SQL / BI tools | Legacy code or libraries that expect RDDs |
| Machine learning with MLlib 2.x+ | mapPartitions-style custom per-partition processing |
| Almost everything in 2024 | Rare edge cases |

**Rule for work: Default to DataFrame. Drop to RDD only when you have no other option.**

---

## The Row Object

DataFrame rows are `Row` objects — named tuples.

```python
row = df.first()
row.name          # access by field name
row['name']       # access by string key
row[0]            # access by index
row.asDict()      # convert to Python dict
```

---

## Interview Cheat Sheet

**Q: What is a Spark DataFrame?**
> A DataFrame is a distributed collection of data organized into named, typed columns — essentially an RDD of Row objects plus a schema, processed through the Catalyst Optimizer and Tungsten execution engine. It's equivalent to a table in a relational database but distributed across a cluster.

**Q: Why is DataFrame faster than RDD in PySpark specifically?**
> PySpark RDDs suffer from Python-JVM serialization overhead — data must cross the language boundary for every operation. DataFrames execute in the JVM regardless of which language you use, avoiding that overhead. Additionally, Catalyst Optimizer rewrites the logical plan for efficiency (predicate pushdown, column pruning), and Tungsten's off-heap binary format reduces GC pressure.

**Q: How is a DataFrame related to an RDD internally?**
> A DataFrame is built on top of an RDD of `Row` objects. You can convert a DataFrame to its underlying RDD with `df.rdd`, and convert an RDD of Row objects back to a DataFrame with `rdd.toDF()` or `spark.createDataFrame(rdd, schema)`. The DataFrame adds schema information and the Catalyst optimization layer on top.

**Q: What is the Tungsten engine?**
> Tungsten is Spark's physical execution engine that works below the DataFrame/Dataset API. It uses off-heap binary memory management (bypassing JVM GC), whole-stage code generation (generates optimized JVM bytecode per query), and cache-aware computation. It typically gives 2-10x speedup over standard JVM execution.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week3 - DataFrame vs RDD") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

## Part 1: Creating DataFrames — 4 Ways

In [ ]:
# Method 1: createDataFrame from list of tuples
data = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob",   "Engineering", 88000),
    (3, "Carol", "Marketing",   72000),
    (4, "Dave",  "Marketing",   68000),
    (5, "Eve",   "Sales",       55000),
]

# With column names only (types inferred)
df1 = spark.createDataFrame(data, ["id", "name", "dept", "salary"])
print("Method 1: createDataFrame with column name list")
df1.show()
df1.printSchema()

In [ ]:
# Method 2: createDataFrame with explicit schema
schema = StructType([
    StructField("id",     IntegerType(), nullable=False),
    StructField("name",   StringType(),  nullable=True),
    StructField("dept",   StringType(),  nullable=True),
    StructField("salary", IntegerType(), nullable=True),
])

df2 = spark.createDataFrame(data, schema)
print("Method 2: createDataFrame with explicit StructType schema")
df2.show()
df2.printSchema()
print(f"Schema difference: df1 uses LongType for int, df2 uses IntegerType (explicit)")

In [ ]:
# Method 3: From list of dicts (convenient but slower)
dict_data = [
    {"id": 1, "name": "Alice", "dept": "Engineering", "salary": 95000},
    {"id": 2, "name": "Bob",   "dept": "Engineering", "salary": 88000},
]
df3 = spark.createDataFrame(dict_data)
print("Method 3: From list of dicts")
df3.show()

# Method 4: From an existing RDD
rdd = sc.parallelize(data)
df4 = rdd.toDF(["id", "name", "dept", "salary"])
print("Method 4: From RDD using toDF()")
df4.show(2)

## Part 2: DataFrame Basics — Inspection Methods

In [ ]:
df = df1

print("=== DataFrame Inspection ===")
print(f"\ndf.count()          = {df.count()}")
print(f"df.columns          = {df.columns}")
print(f"df.dtypes           = {df.dtypes}")
print(f"len(df.columns)     = {len(df.columns)}")
print()

print("df.show() — default 20 rows:")
df.show()

print("df.show(2, truncate=False) — no truncation:")
df.show(2, truncate=False)

print("df.printSchema() — tree view of schema:")
df.printSchema()

print("df.describe() — stats for numeric columns:")
df.describe("salary").show()

In [ ]:
# Accessing the first row
row = df.first()
print("=== Accessing Row Data ===")
print(f"df.first()        = {row}")
print(f"row.name          = {row.name}")
print(f"row['name']       = {row['name']}")
print(f"row[1]            = {row[1]}")
print(f"row.asDict()      = {row.asDict()}")

rows = df.take(2)
print(f"\ndf.take(2)        = {rows}")

## Part 3: DataFrame ↔ RDD Conversion

In [ ]:
# DataFrame → RDD
rdd_from_df = df.rdd
print("DataFrame → RDD:")
print(f"  Type: {type(rdd_from_df)}")
print(f"  First element (Row object): {rdd_from_df.first()}")
print(f"  Access by name: {rdd_from_df.first().name}")
print()

# Common pattern: drop to RDD only for complex operations
high_earners = df.rdd.filter(lambda row: row.salary > 70000)
print(f"  RDD filter result: {high_earners.collect()}")
print()
print("BETTER: do it in DataFrame — let Catalyst optimize:")
df.filter(F.col("salary") > 70000).show()

In [ ]:
# RDD → DataFrame
rdd2 = sc.parallelize([
    (6, "Frank", "Sales",  62000),
    (7, "Grace", "Engineering", 105000),
])

# Using toDF
df_from_rdd1 = rdd2.toDF(["id", "name", "dept", "salary"])
print("RDD → DataFrame via toDF():")
df_from_rdd1.show()

# Using createDataFrame with schema
df_from_rdd2 = spark.createDataFrame(rdd2, schema)
print("RDD → DataFrame via createDataFrame() with schema:")
df_from_rdd2.show()

## Part 4: Performance Proof — DataFrame vs RDD

In [ ]:
import time

# Create a larger dataset
data_large = [(i, f"emp_{i}", ["Eng","Mkt","Sales"][i%3], 50000 + (i*100)) for i in range(50_000)]

# RDD approach
rdd_large = sc.parallelize(data_large, 4)

t0 = time.time()
rdd_result = (
    rdd_large
    .filter(lambda r: r[3] > 70000)          # filter by salary
    .map(lambda r: (r[2], r[3]))              # (dept, salary)
    .reduceByKey(lambda a, b: a + b)          # sum per dept
    .collect()
)
rdd_time = time.time() - t0

# DataFrame approach (same logic)
df_large = spark.createDataFrame(data_large, ["id", "name", "dept", "salary"])

t0 = time.time()
df_result = (
    df_large
    .filter(F.col("salary") > 70000)
    .groupBy("dept")
    .agg(F.sum("salary").alias("total_salary"))
    .collect()
)
df_time = time.time() - t0

print(f"Same computation on 50k rows:")
print(f"  RDD approach:       {rdd_time:.3f}s")
print(f"  DataFrame approach: {df_time:.3f}s")
print(f"  Speedup: {rdd_time/df_time:.1f}x" if df_time > 0 else "")
print(f"\nRDD result:        {sorted(rdd_result)}")
print(f"DataFrame result:  {[(r.dept, r.total_salary) for r in sorted(df_result, key=lambda x: x.dept)]}")

## Part 5: Spark SQL — Query DataFrames Like a Database

In [ ]:
# Register as a temp view → query with SQL
df.createOrReplaceTempView("employees")

result = spark.sql("""
    SELECT dept,
           COUNT(*)        AS headcount,
           AVG(salary)     AS avg_salary,
           MAX(salary)     AS max_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""")

print("SQL query on DataFrame temp view:")
result.show()

print("This produces IDENTICAL results and plans as DataFrame API:")
df.groupBy("dept") \
  .agg(F.count("*").alias("headcount"),
       F.avg("salary").alias("avg_salary"),
       F.max("salary").alias("max_salary")) \
  .orderBy(F.col("avg_salary").desc()) \
  .show()

## Exercises

1. Create a DataFrame of 5 of your favourite movies with columns: `(title, year, genre, rating)`. Use explicit `StructType` schema. Print schema and show all rows.
2. Convert the employees DataFrame to an RDD, extract only `(name, salary)` tuples using `map`, then convert back to a DataFrame.
3. Register the employees DataFrame as a temp view and write a SQL query to find the employee with the highest salary in each department.
4. Use `df.describe()` on the salary column. What are the mean, min, and max values?

In [ ]:
# Exercise 1: Movies DataFrame with explicit schema
movies_schema = StructType([
    StructField("title",  StringType(),  nullable=False),
    StructField("year",   IntegerType(), nullable=False),
    StructField("genre",  StringType(),  nullable=True),
    StructField("rating", DoubleType(),  nullable=True),
])

movies_data = [
    ("Interstellar",   2014, "Sci-Fi",   8.7),
    ("The Dark Knight", 2008, "Action",  9.0),
    ("Inception",       2010, "Sci-Fi",  8.8),
    ("Parasite",        2019, "Thriller", 8.5),
    ("Dune",            2021, "Sci-Fi",  8.0),
]

movies_df = spark.createDataFrame(movies_data, movies_schema)
movies_df.printSchema()
movies_df.show()

In [ ]:
# Exercise 3: SQL - highest salary per dept
df.createOrReplaceTempView("employees")
spark.sql("""
    SELECT dept, name, salary
    FROM (
        SELECT dept, name, salary,
               RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk
        FROM employees
    )
    WHERE rnk = 1
""").show()